# Deep Learning Experiments: MNIST & FashionMNIST

This notebook trains ResNet-18 and ResNet-50 models on MNIST and FashionMNIST datasets.
It uses a 70-10-20 train-val-test split and performs a grid search over specified hyperparameters.

**Requirements:**
- GPU Runtime (Google Colab: Runtime > Change runtime type > GPU)
- PyTorch, Torchvision, Scikit-learn, Pandas, tqdm

In [1]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='torchvision')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, ConcatDataset, random_split
from torchvision import datasets, transforms, models
import pandas as pd
import numpy as np
import time
from tqdm.notebook import tqdm
from IPython.display import display
import gc  # For garbage collection

# Set pandas display options for 6 decimal places
pd.set_option('display.float_format', '{:.6f}'.format)

# Check for GPU and optimize CUDA settings
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if device.type == 'cuda':
    # GPU Optimization Settings
    torch.backends.cudnn.benchmark = True  # Auto-tune convolution algorithms
    torch.backends.cuda.matmul.allow_tf32 = True  # Allow TF32 for faster matmul
    torch.backends.cudnn.allow_tf32 = True  # Allow TF32 for cuDNN
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print("CUDA optimizations enabled: cudnn.benchmark=True, TF32=True")
else:
    print("WARNING: It is highly recommended to run this notebook with a GPU!")

from sklearn.metrics import accuracy_score
from sklearn.svm import SVC

Using device: cuda
GPU: Tesla T4
GPU Memory: 15.83 GB
CUDA optimizations enabled: cudnn.benchmark=True, TF32=True


## 1. Data Preparation
We define a function to load the datasets and split them into 70% Train, 10% Validation, and 20% Test.

In [2]:
def get_dataloaders(dataset_name, batch_size, pin_memory=True, num_workers=0):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)) 
    ])

    if dataset_name == 'MNIST':
        train_part = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
        test_part = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
    elif dataset_name == 'FashionMNIST':
        train_part = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
        test_part = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)
    else:
        raise ValueError("Unknown dataset")

    full_dataset = ConcatDataset([train_part, test_part])
    total_len = len(full_dataset)
    train_len = int(0.7 * total_len)
    val_len = int(0.1 * total_len)
    test_len = total_len - train_len - val_len
    
    generator = torch.Generator().manual_seed(42)
    train_set, val_set, test_set = random_split(full_dataset, [train_len, val_len, test_len], generator=generator)
    
    # DataLoaders with configurable pin_memory
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, pin_memory=pin_memory, num_workers=num_workers)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, pin_memory=pin_memory, num_workers=num_workers)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, pin_memory=pin_memory, num_workers=num_workers)
    
    return train_loader, val_loader, test_loader


## 2. Model Definition
We use ResNet-18 and ResNet-50. Since these models expect 3-channel input (RGB) and our datasets are grayscale (1-channel), we modify the first convolutional layer to accept 1 input channel. Using `weights=None` (no pretrained weights) to avoid the deprecated `pretrained` argument.

In [3]:
def get_model(model_name, num_classes=10, target_device=None):
    """Create ResNet model with memory-optimized settings"""
    if target_device is None:
        target_device = device  # Use global device if not specified
    
    if model_name == 'ResNet18':
        model = models.resnet18(weights=None)
    elif model_name == 'ResNet50':
        model = models.resnet50(weights=None)
    else:
        raise ValueError("Unknown model")
    
    # Modify first layer: 3 input channels -> 1 input channel
    model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    
    # Modify final fully connected layer for number of classes
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    
    return model.to(target_device)

## 3. Training and Evaluation Functions
Includes Automatic Mixed Precision (AMP) for efficiency.

In [4]:
def train_one_epoch(model, loader, optimizer, criterion, scaler, epoch_idx, num_epochs, target_device=None):
    """Train one epoch with memory optimization"""
    if target_device is None:
        target_device = device  # Use global device if not specified
    
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    # Progress bar - set leave=True to keep progress bars visible
    pbar = tqdm(loader, desc=f"Epoch {epoch_idx+1}/{num_epochs} [Train]", leave=True)
    
    for inputs, labels in pbar:
        inputs, labels = inputs.to(target_device, non_blocking=True), labels.to(target_device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)  # Memory optimization: more efficient than zero_grad()
        
        # AMP Context (Updated for new PyTorch versions)
        amp_enabled = (target_device.type == 'cuda')
        with torch.amp.autocast('cuda', enabled=amp_enabled):
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        # Update progress bar
        pbar.set_postfix({'loss': running_loss/total, 'acc': correct/total})
        
    return running_loss / total, correct / total

def evaluate(model, loader, criterion, target_device=None):
    """Evaluate model with memory optimization"""
    if target_device is None:
        target_device = device  # Use global device if not specified
    
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(target_device, non_blocking=True), labels.to(target_device, non_blocking=True)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
    return running_loss / total, correct / total

## 4. Run Experiments
We iterate through the configurations for both MNIST and FashionMNIST.

In [5]:
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, confusion_matrix

# --- Helper function to display training curves as text ---
def display_training_summary(train_losses, train_accs, val_losses, val_accs, config_str):
    """Display training curves as text summary"""
    print(f"\n    📊 Training Summary for: {config_str}")
    print("    " + "-" * 60)
    print(f"    {'Epoch':<8} {'Train Loss':<12} {'Train Acc':<12} {'Val Loss':<12} {'Val Acc':<12}")
    print("    " + "-" * 60)
    for i in range(len(train_losses)):
        print(f"    {i+1:<8} {train_losses[i]:<12.6f} {train_accs[i]:<12.6f} {val_losses[i]:<12.6f} {val_accs[i]:<12.6f}")
    print("    " + "-" * 60)

def display_metrics_summary(metrics_dict, config_str):
    """Display evaluation metrics as text summary"""
    print(f"\n    📈 Evaluation Metrics:")
    print(f"    ├── Accuracy:  {metrics_dict['accuracy']:.6f}")
    print(f"    ├── F1 Score:  {metrics_dict['f1']:.6f}")
    print(f"    ├── Precision: {metrics_dict['precision']:.6f}")
    print(f"    └── Recall:    {metrics_dict['recall']:.6f}")

# --- MAIN EXPERIMENT GRID for Q1(a) ---
import os

# Ensure directories exist
os.makedirs('models', exist_ok=True)

# ============== RESUME SUPPORT ==============
# Load existing results if available
RESUME_FROM = 1  # Set to the experiment number to resume from (1-indexed)
CSV_PATH = 'grid_search_results_q1a.csv'

if os.path.exists(CSV_PATH) and RESUME_FROM > 1:
    print(f"Loading existing results from {CSV_PATH}...")
    df_existing = pd.read_csv(CSV_PATH)
    results = df_existing.to_dict('records')
    print(f"Loaded {len(results)} existing results. Resuming from experiment #{RESUME_FROM}")
else:
    results = []
    RESUME_FROM = 1
# ============================================

# Hyperparameters
datasets_list = ['MNIST', 'FashionMNIST']
models_list = ['ResNet18', 'ResNet50']
batch_sizes = [16, 32]
optimizers_list = ['SGD', 'Adam']
lrs = [0.001, 0.0001]
pin_memories = [False, True]
epochs_list = [2, 5]

# Constant
USE_AMP = True

# Detect Device and apply GPU optimizations
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running on {device}")

if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

start_time_glob = time.time()

# Calculate total number of experiments for progress tracking
total_experiments = (len(datasets_list) * len(batch_sizes) * len(pin_memories) * 
                     len(optimizers_list) * len(lrs) * len(epochs_list) * len(models_list))
current_experiment = 0
print(f"Total experiments to run: {total_experiments}")
print(f"Skipping experiments 1-{RESUME_FROM-1}, starting from #{RESUME_FROM}")
print("=" * 70)

for dataset_name in datasets_list:
    for batch_size in batch_sizes:
        for pin_mem in pin_memories:
            # Prepare Data Loaders once per config
            train_loader, val_loader, test_loader = get_dataloaders(dataset_name, batch_size, pin_memory=pin_mem)
            
            for opt_name in optimizers_list:
                for lr in lrs:
                    for epochs_to_run in epochs_list:
                        for model_name in models_list:
                            current_experiment += 1
                            
                            # Skip already completed experiments
                            if current_experiment < RESUME_FROM:
                                continue
                            
                            progress_pct = (current_experiment / total_experiments) * 100
                            
                            config_str = (f"{dataset_name} | {model_name} | BS={batch_size} | "
                                          f"{opt_name} | LR={lr} | PinMem={pin_mem} | Ep={epochs_to_run}")
                            short_config = f"{dataset_name}_{model_name}_bs{batch_size}_{opt_name}_lr{lr}_pin{pin_mem}_ep{epochs_to_run}"
                            
                            print(f"\n[{current_experiment}/{total_experiments}] ({progress_pct:.1f}%) >>> Running: {config_str}")
                            
                            try:
                                # Init Model
                                model = get_model(model_name).to(device)
                                criterion = nn.CrossEntropyLoss()
                                
                                # Init Optimizer
                                if opt_name == 'SGD':
                                    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
                                else:
                                    optimizer = optim.Adam(model.parameters(), lr=lr)
                                
                                # Scaler for AMP
                                scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
                                
                                # Training history for this config
                                train_losses, train_accs = [], []
                                val_losses, val_accs = [], []
                                
                                # Train
                                t0 = time.time()
                                for epoch in range(epochs_to_run):
                                    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, scaler, epoch, epochs_to_run)
                                    val_loss, val_acc = evaluate(model, val_loader, criterion)
                                    train_losses.append(train_loss)
                                    train_accs.append(train_acc)
                                    val_losses.append(val_loss)
                                    val_accs.append(val_acc)
                                    
                                    # Print epoch summary immediately
                                    print(f"    Epoch {epoch+1}/{epochs_to_run} - Train Loss: {train_loss:.4f}, Train Acc: {100*train_acc:.2f}%, Val Loss: {val_loss:.4f}, Val Acc: {100*val_acc:.2f}%")
                                    
                                duration = time.time() - t0
                                
                                # Final Evaluate on test set
                                test_loss, test_acc = evaluate(model, test_loader, criterion)
                                
                                print(f"    ✅ Done! Val Acc: {val_acc:.6f} | Test Acc: {test_acc:.6f} | Time: {duration:.2f}s")
                                
                                # Calculate F1, Precision, Recall on Test Set
                                model.eval()
                                all_preds = []
                                all_labels = []
                                with torch.no_grad():
                                    for inputs, labels in test_loader:
                                        inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
                                        outputs = model(inputs)
                                        _, preds = torch.max(outputs, 1)
                                        all_preds.extend(preds.cpu().numpy())
                                        all_labels.extend(labels.cpu().numpy())
                                
                                f1 = f1_score(all_labels, all_preds, average='weighted')
                                precision = precision_score(all_labels, all_preds, average='weighted')
                                recall = recall_score(all_labels, all_preds, average='weighted')
                                
                                # --- DISPLAY TRAINING CURVES AS TEXT ---
                                display_training_summary(train_losses, train_accs, val_losses, val_accs, config_str)
                                
                                # --- DISPLAY EVALUATION METRICS AS TEXT ---
                                metrics_dict = {
                                    'accuracy': test_acc,
                                    'f1': f1,
                                    'precision': precision,
                                    'recall': recall
                                }
                                display_metrics_summary(metrics_dict, config_str)
                                
                                # Estimated time remaining
                                elapsed = time.time() - start_time_glob
                                completed_since_resume = current_experiment - RESUME_FROM + 1
                                avg_time_per_exp = elapsed / completed_since_resume
                                remaining_exp = total_experiments - current_experiment
                                eta_seconds = avg_time_per_exp * remaining_exp
                                eta_minutes = eta_seconds / 60
                                print(f"\n    ⏱️  ETA: {eta_minutes:.1f} min remaining ({remaining_exp} experiments left)")
                                
                                # SAVE MODEL
                                save_name = f"{short_config}.pth"
                                save_path = os.path.join('models', save_name)
                                torch.save(model.state_dict(), save_path)
                                print(f"    💾 Saved model to {save_path}")
                                
                                # Store Result
                                results.append({
                                    'Dataset': dataset_name,
                                    'Batch Size': batch_size,
                                    'Optimizer': opt_name,
                                    'Learning Rate': lr,
                                    'Pin Memory': pin_mem,
                                    'Epochs': epochs_to_run,
                                    'Model': model_name,
                                    'Test Accuracy': round(test_acc * 100, 6),
                                    'Training Time (s)': round(duration, 6),
                                    'Model Path': save_path,
                                    'F1 Score': round(f1, 6),
                                    'Precision': round(precision, 6),
                                    'Recall': round(recall, 6)
                                })
                                
                                # Save after each experiment to avoid losing progress
                                df_res = pd.DataFrame(results)
                                df_res.to_csv(CSV_PATH, index=False)
                                
                                # ===== MEMORY CLEANUP =====
                                del model, optimizer, criterion, scaler
                                del train_losses, train_accs, val_losses, val_accs
                                del all_preds, all_labels
                                
                                if torch.cuda.is_available():
                                    torch.cuda.empty_cache()
                                gc.collect()
                                # ==========================
                                
                            except Exception as e:
                                print(f"❌ Error in {config_str}: {e}")
                                import traceback
                                traceback.print_exc()
                                
                                if torch.cuda.is_available():
                                    torch.cuda.empty_cache()
                                gc.collect()
            
            # Clear data loaders
            del train_loader, val_loader, test_loader
            gc.collect()

total_time = time.time() - start_time_glob
print(f"\n{'=' * 70}")
print(f"✅ All Q1(a) Experiments Completed: {current_experiment}/{total_experiments}")
print(f"⏱️  Total Time: {total_time/60:.2f} minutes")
print(f"{'=' * 70}")

Running on cuda
Total experiments to run: 128
Skipping experiments 1-0, starting from #1


100%|██████████| 9.91M/9.91M [00:00<00:00, 41.5MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.17MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 10.5MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.6MB/s]



[1/128] (0.8%) >>> Running: MNIST | ResNet18 | BS=16 | SGD | LR=0.001 | PinMem=False | Ep=2


Epoch 1/2 [Train]:   0%|          | 0/3063 [00:00<?, ?it/s]

    Epoch 1/2 - Train Loss: 0.1673, Train Acc: 94.74%, Val Loss: 0.0565, Val Acc: 98.17%


...

    Epoch 3/5 - Train Loss: 0.4281, Train Acc: 84.23%, Val Loss: 0.4158, Val Acc: 84.66%


Epoch 4/5 [Train]:   0%|          | 0/1532 [00:00<?, ?it/s]

    Epoch 4/5 - Train Loss: 0.3653, Train Acc: 86.52%, Val Loss: 0.3775, Val Acc: 86.43%


Epoch 5/5 [Train]:   0%|          | 0/1532 [00:00<?, ?it/s]

    Epoch 5/5 - Train Loss: 0.3184, Train Acc: 88.12%, Val Loss: 0.3749, Val Acc: 86.29%
    ✅ Done! Val Acc: 0.862857 | Test Acc: 0.864714 | Time: 304.96s

    📊 Training Summary for: FashionMNIST | ResNet50 | BS=32 | Adam | LR=0.0001 | PinMem=True | Ep=5
    ------------------------------------------------------------
    Epoch    Train Loss   Train Acc    Val Loss     Val Acc     
    ------------------------------------------------------------
    1        0.916897     0.664939     0.619303     0.774286    
    2        0.541053     0.804592     0.460780     0.830714    
    3        0.428114     0.842306     0.415817     0.846571    
    4        0.365279     0.865184     0.377474     0.864286    
    5        0.318375     0.881163     0.374886     0.862857    
    ------------------------------------------------------------

    📈 Evaluation Metrics:
    ├── Accuracy:  0.864714
    ├── F1 Score:  0.863509
    ├── Precision: 0.865531
    └── Recall:    0.864714

    ⏱️  ETA: 0.0 m

In [6]:
# --- Summary Tables: Compare ALL Experiments ---
print("=" * 70)
print("Q1(a) SUMMARY - TEXT ANALYSIS")
print("=" * 70)

df_plot = pd.DataFrame(results)

# 1. Best Models Summary
print("\n📊 BEST MODELS BY DATASET AND MODEL TYPE:")
print("-" * 70)
best_models = df_plot.loc[df_plot.groupby(['Dataset', 'Model'])['Test Accuracy'].idxmax()]
for _, row in best_models.iterrows():
    print(f"  {row['Dataset']} + {row['Model']}")
    print(f"    ├── Best Accuracy: {row['Test Accuracy']:.6f}%")
    print(f"    ├── Config: BS={row['Batch Size']}, {row['Optimizer']}, LR={row['Learning Rate']}, Epochs={row['Epochs']}")
    print(f"    └── Training Time: {row['Training Time (s)']:.2f}s")
    print()

# 2. Accuracy Statistics by Model and Dataset
print("\n📈 ACCURACY STATISTICS BY DATASET & MODEL:")
print("-" * 70)
stats = df_plot.groupby(['Dataset', 'Model'])['Test Accuracy'].agg(['mean', 'max', 'min', 'std'])
display(stats.round(6))

# 3. Training Time Statistics
print("\n⏱️  TRAINING TIME STATISTICS:")
print("-" * 70)
time_stats = df_plot.groupby(['Dataset', 'Model'])['Training Time (s)'].agg(['mean', 'min', 'max', 'sum'])
display(time_stats.round(2))

# 4. Accuracy by Hyperparameter
print("\n🔧 MEAN ACCURACY BY HYPERPARAMETER:")
print("-" * 70)

print("\n  By Optimizer:")
opt_stats = df_plot.groupby('Optimizer')['Test Accuracy'].agg(['mean', 'std']).round(6)
display(opt_stats)

print("\n  By Learning Rate:")
lr_stats = df_plot.groupby('Learning Rate')['Test Accuracy'].agg(['mean', 'std']).round(6)
display(lr_stats)

print("\n  By Batch Size:")
bs_stats = df_plot.groupby('Batch Size')['Test Accuracy'].agg(['mean', 'std']).round(6)
display(bs_stats)

print("\n  By Epochs:")
ep_stats = df_plot.groupby('Epochs')['Test Accuracy'].agg(['mean', 'std']).round(6)
display(ep_stats)

# 5. Complete Metrics Summary Table
print("\n📋 COMPLETE METRICS SUMMARY:")
print("-" * 70)
metrics_summary = df_plot.groupby(['Dataset', 'Model']).agg({
    'Test Accuracy': ['mean', 'max', 'std'],
    'F1 Score': ['mean', 'max', 'std'],
    'Precision': ['mean', 'max', 'std'],
    'Recall': ['mean', 'max', 'std'],
    'Training Time (s)': ['mean', 'min', 'max']
}).round(6)
display(metrics_summary)
metrics_summary.to_csv('Q1a_metrics_summary.csv')
print("✅ Saved metrics summary to Q1a_metrics_summary.csv")

Q1(a) SUMMARY - TEXT ANALYSIS

📊 BEST MODELS BY DATASET AND MODEL TYPE:
----------------------------------------------------------------------
  FashionMNIST + ResNet18
    ├── Best Accuracy: 90.764286%
    ├── Config: BS=32, Adam, LR=0.001, Epochs=5
    └── Training Time: 164.19s

  FashionMNIST + ResNet50
    ├── Best Accuracy: 89.400000%
    ├── Config: BS=32, Adam, LR=0.001, Epochs=5
    └── Training Time: 319.76s

  MNIST + ResNet18
    ├── Best Accuracy: 99.014286%
    ├── Config: BS=16, Adam, LR=0.001, Epochs=5
    └── Training Time: 279.89s

  MNIST + ResNet50
    ├── Best Accuracy: 98.700000%
    ├── Config: BS=32, SGD, LR=0.001, Epochs=5
    └── Training Time: 269.34s


📈 ACCURACY STATISTICS BY DATASET & MODEL:
----------------------------------------------------------------------


mean       max       min      std
Dataset      Model                                          
FashionMNIST ResNet18 88.551339 90.764286 85.607143 1.327344
             ResNet50 85.032366 89.400000 78.264286 2.934548
MNIST        ResNet18 98.259821 99.014286 96.357143 0.611813
             ResNet50 96.845982 98.700000 92.957143 1.556477


⏱️  TRAINING TIME STATISTICS:
----------------------------------------------------------------------


mean        min        max         sum
Dataset      Model                                                
FashionMNIST ResNet18 147.970000  59.810000 278.840000 4735.130000
             ResNet50 281.610000 106.740000 552.130000 9011.530000
MNIST        ResNet18 148.630000  60.010000 281.350000 4756.130000
             ResNet50 279.670000 105.760000 550.710000 8949.550000


🔧 MEAN ACCURACY BY HYPERPARAMETER:
----------------------------------------------------------------------

  By Optimizer:


...


  By Epochs:


,mean,std
Epochs,,
2,91.165513,6.290233
5,93.179241,5.227392



📋 COMPLETE METRICS SUMMARY:
----------------------------------------------------------------------


Test Accuracy                    F1 Score           \
                               mean       max      std     mean      max   
Dataset      Model                                                         
FashionMNIST ResNet18     88.551339 90.764286 1.327344 0.885329 0.907080   
             ResNet50     85.032366 89.400000 2.934548 0.848243 0.893539   
MNIST        ResNet18     98.259821 99.014286 0.611813 0.982573 0.990150   
             ResNet50     96.845982 98.700000 1.556477 0.968394 0.986984   

                               Precision                     Recall           \
                           std      mean      max      std     mean      max   
Dataset      Model                                                             
FashionMNIST ResNet18 0.013165  0.887028 0.907552 0.012851 0.885513 0.907643   
             ResNet50 0.031336  0.852383 0.893851 0.029799 0.850324 0.894000   
MNIST        ResNet18 0.006174  0.982814 0.990194 0.005740 0.982598 0.990143   
             ResNet50 0.015699  0.969128 0.987025 0.014661 0.968460 0.987000   

                               Training Time (s)                        
                           std              mean        min        max  
Dataset      Model                                                      
FashionMNIST ResNet18 0.013273        147.972895  59.814627 278.839568  
             ResNet50 0.029345        281.610265 106.742092 552.127002  
MNIST        ResNet18 0.006118        148.628926  60.006474 281.352159  
             ResNet50 0.015565        279.673539 105.760104 550.711372

✅ Saved metrics summary to Q1a_metrics_summary.csv


## 5. Q1(a) Results Analysis
The table below shows the classification accuracy on the test set for all configurations.

In [7]:
df = pd.DataFrame(results)

# Pivot table to match the requested format: Columns for ResNet-18 and ResNet-50
pivot_df = df.pivot_table(index=['Dataset', 'Batch Size', 'Optimizer', 'Learning Rate'], 
                          columns='Model', 
                          values=['Test Accuracy', 'F1 Score', 'Precision', 'Recall']).reset_index()

pivot_df.columns.name = None # Remove 'Model' as name of columns index

display(pivot_df)

pivot_df.to_csv('final_results.csv', index=False)


Dataset Batch Size Optimizer Learning Rate F1 Score           \
Model                                                  ResNet18 ResNet50   
0      FashionMNIST         16      Adam      0.000100 0.882132 0.857597   
1      FashionMNIST         16      Adam      0.001000 0.888686 0.863650   
2      FashionMNIST         16       SGD      0.000100 0.877178 0.828783   
3      FashionMNIST         16       SGD      0.001000 0.894592 0.863930   
4      FashionMNIST         32      Adam      0.000100 0.887590 0.845449   
5      FashionMNIST         32      Adam      0.001000 0.894032 0.850932   
6      FashionMNIST         32       SGD      0.000100 0.866127 0.806371   
7      FashionMNIST         32       SGD      0.001000 0.892294 0.869228   
8             MNIST         16      Adam      0.000100 0.985612 0.968577   
9             MNIST         16      Adam      0.001000 0.985544 0.958191   
10            MNIST         16       SGD      0.000100 0.981351 0.969939   
11            MNIST         16       SGD      0.001000 0.987222 0.982430   
12            MNIST         32      Adam      0.000100 0.982640 0.960808   
13            MNIST         32      Adam      0.001000 0.977164 0.972020   
14            MNIST         32       SGD      0.000100 0.973998 0.952588   
15            MNIST         32       SGD      0.001000 0.987051 0.982600   

      Precision            Recall          Test Accuracy            
Model  ResNet18 ResNet50 ResNet18 ResNet50      ResNet18  ResNet50  
0      0.885637 0.861576 0.881679 0.857643     88.167857 85.764286  
1      0.892502 0.867324 0.888268 0.863018     88.826786 86.301786  
2      0.877474 0.829882 0.877446 0.831893     87.744643 83.189286  
3      0.895773 0.873071 0.894929 0.869071     89.492857 86.907143  
4      0.889066 0.848218 0.888178 0.847732     88.817857 84.773214  
5      0.895294 0.858905 0.894232 0.853232     89.423214 85.323215  
6      0.866896 0.808430 0.866911 0.810250     86.691071 81.025000  
7      0.893579 0.871657 0.892464 0.869750     89.246428 86.975000  
8      0.985672 0.969035 0.985625 0.968572     98.562500 96.857143  
9      0.985654 0.961903 0.985553 0.958607     98.555357 95.860715  
10     0.981391 0.970111 0.981357 0.969982     98.135714 96.998214  
11     0.987305 0.982567 0.987232 0.982447     98.723214 98.244643  
12     0.982782 0.961232 0.982643 0.960839     98.264285 96.083929  
13     0.978563 0.972675 0.977303 0.972018     97.730357 97.201786  
14     0.974064 0.952819 0.974018 0.952607     97.401786 95.260715  
15     0.987078 0.982681 0.987054 0.982607     98.705357 98.260715

### Detailed Analysis for Q1(a)

**Impact of Learning Rate:**
- Higher learning rate (0.001) generally leads to faster convergence but may cause instability
- Lower learning rate (0.0001) provides more stable training but requires more epochs to converge
- Adam optimizer is less sensitive to learning rate choices compared to SGD

**Optimizer Performance (SGD vs Adam):**
- Adam typically achieves higher accuracy in fewer epochs due to adaptive learning rates
- SGD with momentum can match or exceed Adam's performance with proper learning rate tuning
- Adam is more robust to hyperparameter choices

**Model Depth (ResNet18 vs ResNet50):**
- ResNet-50 has more capacity but may overfit on simpler datasets like MNIST
- ResNet-18 often performs comparably or better on MNIST/FashionMNIST due to their simplicity
- Training time significantly increases with ResNet-50

**Batch Size Effects:**
- Larger batch sizes (32) provide more stable gradients but may generalize worse
- Smaller batch sizes (16) introduce more noise which can help escape local minima
- pin_memory=True improves data loading speed when using GPU

**Epochs Analysis:**
- More epochs (5) generally improve accuracy but risk overfitting
- Early stopping based on validation accuracy is recommended
- Training curves help identify optimal stopping point

---
## Q1(b): SVM Classifier on MNIST and FashionMNIST
Train SVM classifiers with different kernels ('poly', 'rbf') and report Testing Classification Accuracy with training time.

In [8]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
import time

def get_flattened_data(dataset_name, max_samples=10000):
    """Load and flatten dataset for SVM (subsample for speed)"""
    transform = transforms.Compose([transforms.ToTensor()])
    
    if dataset_name == 'MNIST':
        train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
        test_data = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
    else:
        train_data = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
        test_data = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)
    
    # Subsample for SVM (full dataset is too slow)
    train_indices = np.random.choice(len(train_data), min(max_samples, len(train_data)), replace=False)
    test_indices = np.random.choice(len(test_data), min(max_samples // 5, len(test_data)), replace=False)
    
    X_train = np.array([train_data[i][0].numpy().flatten() for i in train_indices])
    y_train = np.array([train_data[i][1] for i in train_indices])
    X_test = np.array([test_data[i][0].numpy().flatten() for i in test_indices])
    y_test = np.array([test_data[i][1] for i in test_indices])
    
    # Standardize features
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    return X_train, y_train, X_test, y_test

def display_svm_metrics(metrics_dict, config_str):
    """Display SVM metrics as text summary"""
    print(f"\n    📈 SVM Evaluation Metrics:")
    print(f"    ├── Accuracy:  {metrics_dict['accuracy']:.6f}%")
    print(f"    ├── F1 Score:  {metrics_dict['f1']:.6f}")
    print(f"    ├── Precision: {metrics_dict['precision']:.6f}")
    print(f"    └── Recall:    {metrics_dict['recall']:.6f}")

# SVM Experiments
svm_results = []
svm_kernels = ['poly', 'rbf']
svm_datasets = ['MNIST', 'FashionMNIST']
svm_C_values = [0.1, 1.0, 10.0]  # Regularization parameter
svm_gamma_values = ['scale', 'auto']

print("=" * 60)
print("Q1(b): SVM Classification Experiments")
print("=" * 60)

# Calculate total SVM experiments
total_svm_experiments = len(svm_datasets) * len(svm_kernels) * len(svm_C_values) * len(svm_gamma_values)
current_svm_experiment = 0
svm_start_time = time.time()
print(f"Total SVM experiments to run: {total_svm_experiments}")

for dataset_name in svm_datasets:
    print(f"\n--- Dataset: {dataset_name} ---")
    X_train, y_train, X_test, y_test = get_flattened_data(dataset_name, max_samples=10000)
    print(f"Train samples: {len(X_train)}, Test samples: {len(X_test)}")
    
    for kernel in svm_kernels:
        for C in svm_C_values:
            for gamma in svm_gamma_values:
                current_svm_experiment += 1
                svm_progress_pct = (current_svm_experiment / total_svm_experiments) * 100
                config_str = f"{dataset_name} | kernel={kernel} | C={C} | gamma={gamma}"
                print(f"\n[{current_svm_experiment}/{total_svm_experiments}] ({svm_progress_pct:.1f}%) >>> Running: {config_str}")
                
                try:
                    # Train SVM
                    if kernel == 'poly':
                        svm = SVC(kernel=kernel, C=C, gamma=gamma, degree=3)
                    else:
                        svm = SVC(kernel=kernel, C=C, gamma=gamma)
                    
                    t0 = time.time()
                    svm.fit(X_train, y_train)
                    train_time_ms = (time.time() - t0) * 1000
                    
                    # Predict
                    y_pred = svm.predict(X_test)
                    
                    # Metrics
                    acc = accuracy_score(y_test, y_pred) * 100
                    f1 = f1_score(y_test, y_pred, average='weighted')
                    prec = precision_score(y_test, y_pred, average='weighted')
                    rec = recall_score(y_test, y_pred, average='weighted')
                    
                    print(f"    ✅ Done! Train Time: {train_time_ms:.2f} ms")
                    
                    # --- DISPLAY SVM METRICS AS TEXT ---
                    metrics_dict = {
                        'accuracy': acc,
                        'f1': f1,
                        'precision': prec,
                        'recall': rec
                    }
                    display_svm_metrics(metrics_dict, config_str)
                    
                    # ETA for SVM
                    svm_elapsed = time.time() - svm_start_time
                    svm_avg_time = svm_elapsed / current_svm_experiment
                    svm_remaining = total_svm_experiments - current_svm_experiment
                    svm_eta_min = (svm_avg_time * svm_remaining) / 60
                    print(f"\n    ⏱️  ETA: {svm_eta_min:.1f} min remaining ({svm_remaining} experiments left)")
                    
                    svm_results.append({
                        'Dataset': dataset_name,
                        'Kernel': kernel,
                        'C': C,
                        'Gamma': gamma,
                        'Test Accuracy (%)': round(acc, 6),
                        'F1 Score': round(f1, 6),
                        'Precision': round(prec, 6),
                        'Recall': round(rec, 6),
                        'Training Time (ms)': round(train_time_ms, 6)
                    })
                    
                    # Crash-safe: Save after each experiment
                    pd.DataFrame(svm_results).to_csv('svm_results_q1b.csv', index=False)
                    
                except Exception as e:
                    print(f"❌ Error: {e}")

# SVM Results DataFrame
df_svm = pd.DataFrame(svm_results)
svm_total_time = time.time() - svm_start_time
print(f"\n{'=' * 60}")
print(f"✅ Q1(b) SVM Experiments Completed: {current_svm_experiment}/{total_svm_experiments}")
print(f"⏱️  Total Time: {svm_total_time/60:.2f} minutes")
print("=" * 60)
display(df_svm)
df_svm.to_csv('svm_results_q1b.csv', index=False)

# --- SVM Summary (Text-based) ---
print("\n" + "=" * 60)
print("Q1(b) SVM SUMMARY - TEXT ANALYSIS")
print("=" * 60)

# Best SVM models
print("\n📊 BEST SVM MODELS BY DATASET:")
print("-" * 60)
best_svm = df_svm.loc[df_svm.groupby('Dataset')['Test Accuracy (%)'].idxmax()]
for _, row in best_svm.iterrows():
    print(f"  {row['Dataset']}")
    print(f"    ├── Best Accuracy: {row['Test Accuracy (%)']:.6f}%")
    print(f"    ├── Config: kernel={row['Kernel']}, C={row['C']}, gamma={row['Gamma']}")
    print(f"    └── Training Time: {row['Training Time (ms)']:.2f} ms")
    print()

# Accuracy by Kernel
print("\n📈 MEAN ACCURACY BY KERNEL:")
kernel_stats = df_svm.groupby('Kernel')['Test Accuracy (%)'].agg(['mean', 'max', 'std']).round(6)
display(kernel_stats)

# Accuracy by C value
print("\n📈 MEAN ACCURACY BY C VALUE:")
c_stats = df_svm.groupby('C')['Test Accuracy (%)'].agg(['mean', 'max', 'std']).round(6)
display(c_stats)

# Training time statistics
print("\n⏱️  TRAINING TIME STATISTICS:")
time_stats = df_svm.groupby(['Dataset', 'Kernel'])['Training Time (ms)'].agg(['mean', 'min', 'max']).round(2)
display(time_stats)

print("\n✅ Saved SVM results to svm_results_q1b.csv")

Q1(b): SVM Classification Experiments
Total SVM experiments to run: 24

--- Dataset: MNIST ---
Train samples: 10000, Test samples: 2000

[1/24] (4.2%) >>> Running: MNIST | kernel=poly | C=0.1 | gamma=scale
    ✅ Done! Train Time: 39593.34 ms

    📈 SVM Evaluation Metrics:
    ├── Accuracy:  54.750000%
    ├── F1 Score:  0.578604
    ├── Precision: 0.833002
    └── Recall:    0.547500

    ⏱️  ETA: 18.9 min remaining (23 experiments left)

[2/24] (8.3%) >>> Running: MNIST | kernel=poly | C=0.1 | gamma=auto
    ✅ Done! Train Time: 41302.34 ms

    📈 SVM Evaluation Metrics:
    ├── Accuracy:  38.300000%
    ├── F1 Score:  0.415198
    ├── Precision: 0.841698
    └── Recall:    0.383000

    ⏱️  ETA: 18.0 min remaining (22 experiments left)

[3/24] (12.5%) >>> Running: MNIST | kernel=poly | C=1.0 | gamma=scale
    ✅ Done! Train Time: 22815.17 ms

    📈 SVM Evaluation Metrics:
    ├── Accuracy:  91.500000%
    ├── F1 Score:  0.918552
    ├── Precision: 0.930848
    └── Recall:    0.915000



,Dataset,Kernel,C,Gamma,Test Accuracy (%),F1 Score,Precision,Recall,Training Time (ms)
0,MNIST,poly,0.100000,scale,54.750000,0.578604,0.833002,0.547500,39593.338728
1,MNIST,poly,0.100000,auto,38.300000,0.415198,0.841698,0.383000,41302.342892
2,MNIST,poly,1.000000,scale,91.500000,0.918552,0.930848,0.915000,22815.172195
3,MNIST,poly,1.000000,auto,87.950000,0.887632,0.915480,0.879500,25878.817797
4,MNIST,poly,10.000000,scale,95.500000,0.955114,0.955722,0.955000,16029.763937
5,MNIST,poly,10.000000,auto,95.350000,0.953705,0.954689,0.953500,16236.431122
6,MNIST,rbf,0.100000,scale,89.350000,0.894577,0.899443,0.893500,20014.378786
7,MNIST,rbf,0.100000,auto,89.850000,0.898856,0.901761,0.898500,19743.942261
8,MNIST,rbf,1.000000,scale,94.850000,0.948441,0.948676,0.948500,11361.692667
9,MNIST,rbf,1.000000,auto,95.200000,0.951952,0.952157,0.952000,10717.605114



Q1(b) SVM SUMMARY - TEXT ANALYSIS

📊 BEST SVM MODELS BY DATASET:
------------------------------------------------------------
  FashionMNIST
    ├── Best Accuracy: 86.300000%
    ├── Config: kernel=rbf, C=10.0, gamma=scale
    └── Training Time: 9091.31 ms

  MNIST
    ├── Best Accuracy: 95.600000%
    ├── Config: kernel=rbf, C=10.0, gamma=scale
    └── Training Time: 10730.04 ms


📈 MEAN ACCURACY BY KERNEL:


,mean,max,std
Kernel,,,
poly,78.004167,95.500000,17.325946
rbf,88.358333,95.600000,6.148571



📈 MEAN ACCURACY BY C VALUE:


,mean,max,std
C,,,
0.100000,70.956250,89.850000,17.525623
1.000000,87.975000,95.200000,5.311107
10.000000,90.612500,95.600000,5.243482



⏱️  TRAINING TIME STATISTICS:


mean          min          max
Dataset      Kernel                                       
FashionMNIST poly   13998.950000  9229.090000 21332.990000
             rbf    10984.110000  9032.870000 14829.970000
MNIST        poly   26975.980000 16029.760000 41302.340000
             rbf    13723.880000  9775.640000 20014.380000


✅ Saved SVM results to svm_results_q1b.csv


In [9]:
%pip install thop

Note: you may need to restart the kernel to use updated packages.


---
## Q2: CPU vs GPU Performance Comparison (FashionMNIST)
Compare training time, FLOPs, and classification accuracy on CPU vs GPU for FashionMNIST dataset.

In [10]:
# Install thop for FLOPs calculation if not already installed
try:
    from thop import profile, clever_format
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'thop'])
    from thop import profile, clever_format

import copy

def calculate_flops(model, input_size=(1, 1, 28, 28)):
    """Calculate FLOPs for a model (uses a copy to avoid modifying original model)"""
    # Create a deep copy on CPU for FLOPs calculation to avoid moving original model
    model_copy = copy.deepcopy(model).cpu()
    model_copy.eval()
    dummy_input = torch.randn(input_size)
    flops, params = profile(model_copy, inputs=(dummy_input,), verbose=False)
    del model_copy  # Clean up
    gc.collect()
    return flops, params

def display_q2_training_summary(train_losses, train_accs, val_losses, val_accs, config_str):
    """Display Q2 training curves as text summary"""
    print(f"\n    📊 Training Summary:")
    print("    " + "-" * 60)
    print(f"    {'Epoch':<8} {'Train Loss':<12} {'Train Acc':<12} {'Val Loss':<12} {'Val Acc':<12}")
    print("    " + "-" * 60)
    for i in range(len(train_losses)):
        print(f"    {i+1:<8} {train_losses[i]:<12.6f} {train_accs[i]:<12.6f} {val_losses[i]:<12.6f} {val_accs[i]:<12.6f}")
    print("    " + "-" * 60)

def display_q2_metrics(metrics_dict, config_str):
    """Display Q2 evaluation metrics as text summary"""
    print(f"\n    📈 Evaluation Metrics:")
    print(f"    ├── Accuracy:  {metrics_dict['accuracy']:.6f}")
    print(f"    ├── F1 Score:  {metrics_dict['f1']:.6f}")
    print(f"    ├── Precision: {metrics_dict['precision']:.6f}")
    print(f"    └── Recall:    {metrics_dict['recall']:.6f}")

def train_and_evaluate_device(model_name, device_type, train_loader, val_loader, test_loader, 
                               optimizer_name, lr, epochs, use_amp=True):
    """Train model on specified device and return metrics with memory optimization"""
    # Create device object for this specific run
    target_device = torch.device(device_type)
    
    # Get model on the target device (pass target_device explicitly)
    model = get_model(model_name, target_device=target_device)
    criterion = nn.CrossEntropyLoss()
    
    if optimizer_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    else:
        optimizer = optim.Adam(model.parameters(), lr=lr)
    
    # AMP only for CUDA
    amp_enabled = use_amp and device_type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=amp_enabled)
    
    # Calculate FLOPs (uses a copy, doesn't affect original model)
    flops, params = calculate_flops(model)
    
    # Train with tracking
    train_losses, train_accs, val_losses, val_accs = [], [], [], []
    t0 = time.time()
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        # Progress bar for training batches - set leave=True to keep visible
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [{device_type.upper()}]", leave=True)
        
        for inputs, labels in pbar:
            inputs, labels = inputs.to(target_device, non_blocking=True), labels.to(target_device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)  # Memory optimization: set_to_none=True
            
            if amp_enabled:
                with torch.amp.autocast('cuda'):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            # Update progress bar with current metrics
            pbar.set_postfix({
                'loss': f'{running_loss/total:.4f}',
                'acc': f'{100*correct/total:.2f}%'
            })
        
        train_loss = running_loss / total
        train_acc = correct / total
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        
        # Validation - pass target_device explicitly
        val_loss, val_acc = evaluate(model, val_loader, criterion, target_device=target_device)
        val_losses.append(val_loss)
        val_accs.append(val_acc)
        
        # Print epoch summary
        print(f"    Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, Train Acc: {100*train_acc:.2f}%, Val Acc: {100*val_acc:.2f}%")
    
    train_time_ms = (time.time() - t0) * 1000
    
    # Test evaluation with predictions for metrics
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc="Testing", leave=False):
            inputs, labels = inputs.to(target_device, non_blocking=True), labels.to(target_device, non_blocking=True)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    test_acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')
    
    # Memory cleanup
    result = {
        'test_acc': test_acc * 100,
        'train_time_ms': train_time_ms,
        'flops': flops,
        'params': params,
        'train_losses': train_losses,
        'train_accs': train_accs,
        'val_losses': val_losses,
        'val_accs': val_accs,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }
    
    # Cleanup model and optimizer
    del model, optimizer, criterion, scaler
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    
    return result

# Q2 Experiments: CPU vs GPU comparison
print("=" * 70)
print("Q2: CPU vs GPU Performance Comparison on FashionMNIST")
print("=" * 70)

q2_results = []
q2_models = ['ResNet18', 'ResNet50']
q2_optimizers = ['SGD', 'Adam']
q2_batch_size = 32  # Increased batch size for better GPU utilization
q2_lr = 0.001
q2_epochs = 2  # Reduced epochs for CPU (can increase for final run)

# Check available devices
devices_to_test = ['cpu']
if torch.cuda.is_available():
    devices_to_test.append('cuda')
    # GPU optimizations
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    print("CUDA optimizations enabled: cudnn.benchmark=True, TF32=True")
else:
    print("WARNING: No GPU available, only CPU results will be generated")

print("=" * 70)

# Calculate total Q2 experiments
total_q2_experiments = len(devices_to_test) * len(q2_models) * len(q2_optimizers)
current_q2_experiment = 0
q2_start_time = time.time()
print(f"Total Q2 experiments to run: {total_q2_experiments}")

for device_type in devices_to_test:
    # Create dataloaders optimized for each device type
    # pin_memory=True only helps with GPU, use False for CPU to avoid overhead
    use_pin_memory = (device_type == 'cuda')
    train_loader, val_loader, test_loader = get_dataloaders('FashionMNIST', q2_batch_size, pin_memory=use_pin_memory)
    print(f"\n--- Device: {device_type.upper()} (pin_memory={use_pin_memory}) ---")
    
    for model_name in q2_models:
        for opt_name in q2_optimizers:
            current_q2_experiment += 1
            q2_progress_pct = (current_q2_experiment / total_q2_experiments) * 100
            config_str = f"{device_type.upper()} | {model_name} | {opt_name} | LR={q2_lr}"
            print(f"\n[{current_q2_experiment}/{total_q2_experiments}] ({q2_progress_pct:.1f}%) >>> Running: {config_str}")
            
            try:
                result = train_and_evaluate_device(
                    model_name, device_type, train_loader, val_loader, test_loader,
                    opt_name, q2_lr, q2_epochs, use_amp=True
                )
                
                flops_formatted, params_formatted = clever_format([result['flops'], result['params']], "%.2f")
                
                print(f"\n    ✅ Done! Test Acc: {result['test_acc']:.6f}% | Train Time: {result['train_time_ms']:.2f} ms | FLOPs: {flops_formatted}")
                
                # --- DISPLAY TRAINING CURVES AS TEXT ---
                display_q2_training_summary(
                    result['train_losses'], result['train_accs'], 
                    result['val_losses'], result['val_accs'],
                    config_str
                )
                
                # --- DISPLAY EVALUATION METRICS AS TEXT ---
                metrics_dict = {
                    'accuracy': result['test_acc'] / 100,
                    'f1': result['f1'],
                    'precision': result['precision'],
                    'recall': result['recall']
                }
                display_q2_metrics(metrics_dict, config_str)
                
                # ETA for Q2
                q2_elapsed = time.time() - q2_start_time
                q2_avg_time = q2_elapsed / current_q2_experiment
                q2_remaining = total_q2_experiments - current_q2_experiment
                q2_eta_min = (q2_avg_time * q2_remaining) / 60
                print(f"\n    ⏱️  ETA: {q2_eta_min:.1f} min remaining ({q2_remaining} experiments left)")
                
                q2_results.append({
                    'Compute': device_type.upper(),
                    'Batch Size': q2_batch_size,
                    'Optimizer': opt_name,
                    'Learning Rate': q2_lr,
                    'Model': model_name,
                    'Test Accuracy (%)': round(result['test_acc'], 6),
                    'Training Time (ms)': round(result['train_time_ms'], 6),
                    'FLOPs': flops_formatted,
                    'Parameters': params_formatted,
                    'F1 Score': round(result['f1'], 6),
                    'Precision': round(result['precision'], 6),
                    'Recall': round(result['recall'], 6)
                })
                
                # Crash-safe: Save after each experiment
                pd.DataFrame(q2_results).to_csv('cpu_gpu_comparison_q2.csv', index=False)
                
                # Memory cleanup after each experiment
                del result
                gc.collect()
                
            except Exception as e:
                print(f"❌ Error: {e}")
                import traceback
                traceback.print_exc()
                # Cleanup on error
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                gc.collect()
    
    # Cleanup data loaders after all experiments for this device
    del train_loader, val_loader, test_loader
    gc.collect()

# Q2 Results DataFrame
df_q2 = pd.DataFrame(q2_results)
q2_total_time = time.time() - q2_start_time
print(f"\n{'=' * 70}")
print(f"✅ Q2 Experiments Completed: {current_q2_experiment}/{total_q2_experiments}")
print(f"⏱️  Total Time: {q2_total_time/60:.2f} minutes")
print("=" * 70)
display(df_q2)
df_q2.to_csv('cpu_gpu_comparison_q2.csv', index=False)

# --- Q2 Summary (Text-based) ---
print("\n" + "=" * 70)
print("Q2 SUMMARY - CPU vs GPU COMPARISON")
print("=" * 70)

# Training time comparison
print("\n⏱️  TRAINING TIME COMPARISON:")
print("-" * 70)
time_pivot = df_q2.pivot_table(index='Model', columns='Compute', values='Training Time (ms)', aggfunc='mean')
display(time_pivot.round(2))

if 'CPU' in time_pivot.columns and 'CUDA' in time_pivot.columns:
    print("\n📈 GPU SPEEDUP FACTOR:")
    speedup = time_pivot['CPU'] / time_pivot['CUDA']
    for model in speedup.index:
        print(f"    {model}: {speedup[model]:.2f}x faster on GPU")

# Accuracy comparison
print("\n🎯 ACCURACY COMPARISON:")
print("-" * 70)
acc_pivot = df_q2.pivot_table(index='Model', columns='Compute', values='Test Accuracy (%)', aggfunc='mean')
display(acc_pivot.round(6))

# Best configurations
print("\n🏆 BEST CONFIGURATIONS:")
print("-" * 70)
best_q2 = df_q2.loc[df_q2.groupby('Compute')['Test Accuracy (%)'].idxmax()]
for _, row in best_q2.iterrows():
    print(f"  {row['Compute']}")
    print(f"    ├── Best Accuracy: {row['Test Accuracy (%)']:.6f}%")
    print(f"    ├── Model: {row['Model']}, Optimizer: {row['Optimizer']}")
    print(f"    ├── Training Time: {row['Training Time (ms)']:.2f} ms")
    print(f"    └── FLOPs: {row['FLOPs']}")
    print()

# Full metrics summary
print("\n📋 FULL METRICS SUMMARY:")
print("-" * 70)
metrics_summary = df_q2.groupby(['Compute', 'Model']).agg({
    'Test Accuracy (%)': 'mean',
    'Training Time (ms)': 'mean',
    'F1 Score': 'mean',
    'Precision': 'mean',
    'Recall': 'mean'
}).round(6)
display(metrics_summary)

print("\n✅ Saved Q2 results to cpu_gpu_comparison_q2.csv")

Q2: CPU vs GPU Performance Comparison on FashionMNIST
GPU available: Tesla T4
CUDA optimizations enabled: cudnn.benchmark=True, TF32=True
Total Q2 experiments to run: 8

--- Device: CPU (pin_memory=False) ---

[1/8] (12.5%) >>> Running: CPU | ResNet18 | SGD | LR=0.001


Epoch 1/2 [CPU]:   0%|          | 0/1532 [00:00<?, ?it/s]

    Epoch 1/2 - Train Loss: 0.4965, Train Acc: 82.12%, Val Acc: 86.77%


Epoch 2/2 [CPU]:   0%|          | 0/1532 [00:00<?, ?it/s]

    Epoch 2/2 - Train Loss: 0.3254, Train Acc: 88.03%, Val Acc: 88.96%


...


📈 GPU SPEEDUP FACTOR:
    ResNet18: 6.12x faster on GPU
    ResNet50: 9.88x faster on GPU

🎯 ACCURACY COMPARISON:
----------------------------------------------------------------------


Compute,CPU,CUDA
Model,,
ResNet18,88.382142,87.057143
ResNet50,85.975000,85.571428



🏆 BEST CONFIGURATIONS:
----------------------------------------------------------------------
  CPU
    ├── Best Accuracy: 88.535714%
    ├── Model: ResNet18, Optimizer: SGD
    ├── Training Time: 341621.31 ms
    └── FLOPs: 33.18M

  CUDA
    ├── Best Accuracy: 87.971429%
    ├── Model: ResNet18, Optimizer: SGD
    ├── Training Time: 62967.30 ms
    └── FLOPs: 33.18M


📋 FULL METRICS SUMMARY:
----------------------------------------------------------------------


Test Accuracy (%)  Training Time (ms)  F1 Score  Precision  \
Compute Model                                                                  
CPU     ResNet18          88.382142       406212.338805  0.884085   0.885506   
        ResNet50          85.975000      1172567.770124  0.859552   0.864286   
CUDA    ResNet18          87.057143        66380.471468  0.868134   0.880508   
        ResNet50          85.571428       118671.958804  0.852386   0.858238   

                   Recall  
Compute Model              
CPU     ResNet18 0.883822  
        ResNet50 0.859750  
CUDA    ResNet18 0.870572  
        ResNet50 0.855714


✅ Saved Q2 results to cpu_gpu_comparison_q2.csv


---
## Final Results Summary
Tables formatted as per assignment requirements.

In [11]:
# --- Q1(a) Table: Test Classification Accuracy (in %) ---
print("=" * 80)
print("Q1(a) RESULTS TABLE: Test Classification Accuracy (in %)")
print("=" * 80)

# Filter results for the required table format
df_q1a = pd.DataFrame(results)

# Create pivot table for each dataset
for dataset in ['MNIST', 'FashionMNIST']:
    print(f"\n### {dataset} Dataset ###")
    df_dataset = df_q1a[df_q1a['Dataset'] == dataset]
    
    # Aggregate by taking max accuracy across pin_memory and epochs variations
    pivot = df_dataset.pivot_table(
        index=['Batch Size', 'Optimizer', 'Learning Rate'],
        columns='Model',
        values='Test Accuracy',
        aggfunc='max'
    ).reset_index()
    
    # Reorder columns
    pivot = pivot[['Batch Size', 'Optimizer', 'Learning Rate', 'ResNet18', 'ResNet50']]
    display(pivot)
    pivot.to_csv(f'q1a_{dataset}_results.csv', index=False)

# --- Q1(b) Table: SVM Results ---
print("\n" + "=" * 80)
print("Q1(b) RESULTS TABLE: SVM Classification")
print("=" * 80)

for dataset in ['MNIST', 'FashionMNIST']:
    print(f"\n### {dataset} Dataset ###")
    df_svm_dataset = df_svm[df_svm['Dataset'] == dataset]
    pivot_svm = df_svm_dataset.pivot_table(
        index=['Kernel', 'C', 'Gamma'],
        values=['Test Accuracy (%)', 'Training Time (ms)'],
        aggfunc='first'
    ).reset_index()
    display(pivot_svm)

# --- Q2 Table: CPU vs GPU Comparison ---
print("\n" + "=" * 80)
print("Q2 RESULTS TABLE: CPU vs GPU Comparison (FashionMNIST)")
print("=" * 80)

pivot_q2 = df_q2.pivot_table(
    index=['Compute', 'Batch Size', 'Optimizer', 'Learning Rate'],
    columns='Model',
    values=['Test Accuracy (%)', 'Training Time (ms)', 'FLOPs'],
    aggfunc='first'
).reset_index()

display(pivot_q2)
pivot_q2.to_csv('q2_cpu_gpu_results.csv', index=False)

# --- Save Best Model ---
print("\n" + "=" * 80)
print("BEST MODEL SELECTION")
print("=" * 80)

# Find best model from Q1(a)
best_idx = df_q1a['Test Accuracy'].idxmax()
best_model_info = df_q1a.loc[best_idx]
print(f"\nBest Model Configuration:")
print(f"  Dataset: {best_model_info['Dataset']}")
print(f"  Model: {best_model_info['Model']}")
print(f"  Batch Size: {best_model_info['Batch Size']}")
print(f"  Optimizer: {best_model_info['Optimizer']}")
print(f"  Learning Rate: {best_model_info['Learning Rate']}")
print(f"  Epochs: {best_model_info['Epochs']}")
print(f"  Test Accuracy: {best_model_info['Test Accuracy']}%")
print(f"  Model Path: {best_model_info['Model Path']}")

Q1(a) RESULTS TABLE: Test Classification Accuracy (in %)

### MNIST Dataset ###


Model,Batch Size,Optimizer,Learning Rate,ResNet18,ResNet50
0,16,Adam,0.000100,98.757143,97.742857
1,16,Adam,0.001000,99.014286,98.314286
2,16,SGD,0.000100,98.478571,97.907143
3,16,SGD,0.001000,98.950000,98.550000
4,32,Adam,0.000100,98.657143,97.021429
5,32,Adam,0.001000,98.778571,98.535714
6,32,SGD,0.000100,97.914286,96.871429
7,32,SGD,0.001000,98.821429,98.700000



### FashionMNIST Dataset ###


Model,Batch Size,Optimizer,Learning Rate,ResNet18,ResNet50
0,16,Adam,0.000100,89.350000,87.914286
1,16,Adam,0.001000,90.571429,87.421429
2,16,SGD,0.000100,88.657143,85.600000
3,16,SGD,0.001000,90.064286,89.021429
4,32,Adam,0.000100,89.685714,86.907143
5,32,Adam,0.001000,90.764286,89.400000
6,32,SGD,0.000100,87.864286,83.600000
7,32,SGD,0.001000,89.928571,89.292857



Q1(b) RESULTS TABLE: SVM Classification

### MNIST Dataset ###


...


### FashionMNIST Dataset ###


,Kernel,C,Gamma,Test Accuracy (%),Training Time (ms)
0,poly,0.100000,auto,69.050000,21332.985163
1,poly,0.100000,scale,69.050000,21222.198963
2,poly,1.000000,auto,82.150000,11507.883310
3,poly,1.000000,scale,82.150000,11343.184471
4,poly,10.000000,auto,85.150000,9229.094982
5,poly,10.000000,scale,85.150000,9358.361483
6,rbf,0.100000,auto,78.650000,14829.966784
7,rbf,0.100000,scale,78.650000,14453.416824
8,rbf,1.000000,auto,85.000000,9144.469976
9,rbf,1.000000,scale,85.000000,9352.647066



Q2 RESULTS TABLE: CPU vs GPU Comparison (FashionMNIST)


Compute Batch Size Optimizer Learning Rate    FLOPs           \
Model                                            ResNet18 ResNet50   
0         CPU         32      Adam      0.001000   33.18M   78.76M   
1         CPU         32       SGD      0.001000   33.18M   78.76M   
2        CUDA         32      Adam      0.001000   33.18M   78.76M   
3        CUDA         32       SGD      0.001000   33.18M   78.76M   

      Test Accuracy (%)           Training Time (ms)                 
Model          ResNet18  ResNet50           ResNet18       ResNet50  
0             88.228571 87.114286      470803.363562 1281027.967453  
1             88.535714 84.835714      341621.314049 1064107.572794  
2             86.142857 86.964286       69793.638468  125692.883253  
3             87.971429 84.178571       62967.304468  111651.034355


BEST MODEL SELECTION

Best Model Configuration:
  Dataset: MNIST
  Model: ResNet18
  Batch Size: 16
  Optimizer: Adam
  Learning Rate: 0.001
  Epochs: 5
  Test Accuracy: 99.014286%
  Model Path: models/MNIST_ResNet18_bs16_Adam_lr0.001_pinFalse_ep5.pth


### Detailed Analysis for Q2: CPU vs GPU Comparison

**Training Time Analysis:**
- GPU training is significantly faster (typically 10-50x) compared to CPU
- The speedup is more pronounced for larger models (ResNet-50 vs ResNet-18)
- AMP (Automatic Mixed Precision) provides additional speedup on GPU with minimal accuracy loss

**FLOPs Analysis:**
- ResNet-50 has approximately 4x more FLOPs than ResNet-18
- FLOPs count remains constant regardless of CPU/GPU (it's a model property)
- Higher FLOPs correlate with longer training times

**Classification Accuracy:**
- Accuracy should be similar between CPU and GPU (same model, same data)
- Minor differences may occur due to floating-point precision differences
- GPU with AMP may show slight variations due to mixed precision

**Recommendations:**
- Always use GPU when available for deep learning training
- Use AMP for additional speedup without significant accuracy loss
- For inference on edge devices, consider model quantization or pruning